# Model & Trading Performance

Tracks Brier score, feature importances, and live trade outcomes over time.

Data sources:
- `logs/model_metrics.jsonl` — appended every `models/train.py` run
- `logs/resolved_trades.csv` — appended each time a position resolves on Kalshi

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

ROOT = Path('.').resolve().parent
METRICS_PATH  = ROOT / 'logs' / 'model_metrics.jsonl'
RESOLVED_PATH = ROOT / 'logs' / 'resolved_trades.csv'

## Training Metrics Over Time

In [ ]:
if not METRICS_PATH.exists():
    print('No model_metrics.jsonl yet — run models/train.py first')
else:
    runs = [json.loads(l) for l in METRICS_PATH.read_text().strip().splitlines()]
    mdf = pd.DataFrame(runs)
    mdf['timestamp'] = pd.to_datetime(mdf['timestamp'])
    mdf['run'] = range(1, len(mdf) + 1)
    print(f'Training runs logged: {len(mdf)}')
    display(mdf[['timestamp', 'brier_score', 'log_loss', 'n_train', 'n_test']])

In [ ]:
if 'mdf' in dir() and len(mdf) > 1:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    ax = axes[0]
    ax.plot(mdf['run'], mdf['brier_score'], marker='o', color='steelblue', linewidth=2)
    ax.axhline(0.20, color='red', linestyle='--', linewidth=1, label='Target (0.20)')
    ax.set_xlabel('Training Run'); ax.set_ylabel('Brier Score')
    ax.set_title('Brier Score Over Retrains'); ax.legend()
    ax = axes[1]
    ax.plot(mdf['run'], mdf['n_train'], marker='o', color='darkorange', linewidth=2, label='Train rows')
    ax.plot(mdf['run'], mdf['n_test'], marker='s', color='mediumseagreen', linewidth=2, label='Test rows')
    ax.set_xlabel('Training Run'); ax.set_ylabel('Rows')
    ax.set_title('Dataset Size Over Retrains'); ax.legend()
    plt.tight_layout(); plt.show()
elif 'mdf' in dir():
    print('Only one run so far — retrain again to see trends')

## Feature Importance Drift

Watch `sentiment_score` (orange) rise as more correctly-labeled snapshots accumulate.

In [ ]:
if 'mdf' in dir():
    fi_rows = []
    for _, row in mdf.iterrows():
        for feat, imp in row['feature_importances'].items():
            fi_rows.append({'run': row['run'], 'timestamp': row['timestamp'], 'feature': feat, 'importance': imp})
    fi = pd.DataFrame(fi_rows)

    latest = fi[fi['run'] == fi['run'].max()].sort_values('importance', ascending=True)
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ['darkorange' if 'sentiment' in f else 'steelblue' for f in latest['feature']]
    ax.barh(latest['feature'], latest['importance'], color=colors, alpha=0.85)
    ax.set_xlabel('Feature Importance')
    ax.set_title(f"Feature Importances — Run {fi['run'].max()} (orange = sentiment)")
    plt.tight_layout(); plt.show()

    if len(mdf) > 1:
        fig, ax = plt.subplots(figsize=(10, 5))
        for feat in ['market_price', 'sentiment_score', 'precip_prob_new_york', 'btc_change_1h', 'volume_24h']:
            sub = fi[fi['feature'] == feat]
            ax.plot(sub['run'], sub['importance'], marker='o', label=feat, linewidth=2)
        ax.set_xlabel('Training Run'); ax.set_ylabel('Importance')
        ax.set_title('Feature Importance Drift Over Retrains'); ax.legend()
        plt.tight_layout(); plt.show()

## Live Trade Outcomes

In [ ]:
if not RESOLVED_PATH.exists():
    print('No resolved_trades.csv yet — positions settle when Kalshi finalizes contracts')
else:
    rdf = pd.read_csv(RESOLVED_PATH, parse_dates=['timestamp'])
    rdf['won'] = rdf['won'].astype(str).str.lower() == 'true'
    rdf = rdf.sort_values('timestamp').reset_index(drop=True)
    rdf['cumulative_pnl'] = rdf['pnl_dollars'].cumsum()
    rdf['trade_num'] = rdf.index + 1
    win_rate = rdf['won'].mean()
    total_pnl = rdf['pnl_dollars'].sum()
    print(f'Resolved: {len(rdf)} trades  |  Win rate: {win_rate:.1%}  |  Total P&L: ${total_pnl:+.2f}')
    display(rdf[['timestamp', 'contract_id', 'side', 'result', 'won', 'pnl_dollars', 'cumulative_pnl']].tail(10))

In [ ]:
if 'rdf' in dir() and len(rdf) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    ax = axes[0]
    ax.plot(rdf['trade_num'], rdf['cumulative_pnl'], marker='o', linewidth=2, color='steelblue')
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.fill_between(rdf['trade_num'], rdf['cumulative_pnl'], 0,
                    where=rdf['cumulative_pnl'] >= 0, alpha=0.15, color='green')
    ax.fill_between(rdf['trade_num'], rdf['cumulative_pnl'], 0,
                    where=rdf['cumulative_pnl'] < 0, alpha=0.15, color='red')
    ax.set_xlabel('Trade #'); ax.set_ylabel('Cumulative P&L ($)')
    ax.set_title('Cumulative P&L — Resolved Trades')
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.2f'))

    def cat(cid):
        p = cid.split('-')[0]
        if p in {'KXBTCD','KXBTC15M','KXETH15M','KXSOL15M','KXXRP15M','KXDOGE','KXDOGE15M','KXBNB15M'}: return 'crypto'
        if 'HIGH' in p or 'LOWT' in p: return 'weather'
        if p in {'KXMLB','KXMLBHRR','KXNBA','KXNHL','KXF1'}: return 'sports'
        if p in {'KXFED','KXCPI','KXGDP','KXADP','KXWTI','KXEURUSD','KXUSDJPY'}: return 'macro'
        return 'other'
    rdf['category'] = rdf['contract_id'].apply(cat)
    cat_stats = rdf.groupby('category').agg(
        n=('won', 'count'), win_rate=('won', 'mean'), pnl=('pnl_dollars', 'sum')
    ).reset_index()

    ax = axes[1]
    palette = ['steelblue', 'darkorange', 'mediumseagreen', 'mediumpurple', 'gray']
    ax.bar(cat_stats['category'], cat_stats['win_rate'], color=palette[:len(cat_stats)], alpha=0.85)
    ax.axhline(0.52, color='red', linestyle='--', linewidth=1.2, label='Gate (52%)')
    ax.set_ylim(0, 1); ax.set_ylabel('Win Rate')
    ax.set_title('Win Rate by Market Category')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1)); ax.legend()
    plt.tight_layout(); plt.show()
    print(cat_stats.to_string(index=False))